# Knowledge Bank Creation and Management

## Learning Objectives

By completing this notebook, you will:
1. Create a Knowledge Bank from documents and datasets
2. Understand chunking strategies and their impact
3. Configure embeddings and vector stores
4. Add metadata for filtered retrieval
5. Maintain and update Knowledge Banks over time

## Prerequisites

- Module 0 completed (LLM Mesh configured)
- Module 1 completed (Prompt design basics)
- Understanding of vector embeddings (conceptual)
- Sample documents for testing

## Estimated Time: 60 minutes

---

In [ ]:
learning_objectives(["Create a Knowledge Bank from documents and datasets", "Understand chunking strategies and their impact", "Configure embeddings and vector stores", "Add metadata for filtered retrieval", "Maintain and update Knowledge Banks over time"])

## What is a Knowledge Bank?

**Knowledge Banks** are Dataiku's managed RAG (Retrieval-Augmented Generation) infrastructure:

```
Documents → Chunking → Embedding → Vector Store → Retrieval
```

### The RAG Pipeline

1. **Ingestion**: Load documents from files, datasets, or APIs
2. **Chunking**: Split into semantic units (~500 tokens)
3. **Embedding**: Convert text to vectors (1536 dimensions)
4. **Indexing**: Store in vector database for fast search
5. **Retrieval**: Find relevant chunks for user queries
6. **Generation**: Use chunks as context for LLM

### Why Use Knowledge Banks?

**Without Knowledge Banks**:
- LLMs are limited to training data (outdated)
- Cannot answer questions about your proprietary documents
- Hallucinate when uncertain
- No grounding in factual sources

**With Knowledge Banks**:
- Ground responses in your documents
- Always current (update the KB)
- Cite sources for transparency
- Reduce hallucinations dramatically

## Setup

Import libraries for Knowledge Bank creation.

In [ ]:
import sys; sys.path.insert(0, '../../../../..')
from resources.notebook_style import apply_course_theme, learning_objectives, section_divider, callout, key_takeaways
from resources.graphics.plot_theme import apply_plot_theme

In [ ]:
# Apply course styling
apply_course_theme()
apply_plot_theme()

In [ ]:
import dataiku
from dataiku.knowledge_bank import KnowledgeBank
from dataiku.llm import LLM
import pandas as pd
from typing import List, Dict
import json

## Exercise 1: Create Your First Knowledge Bank

We'll create a Knowledge Bank from sample commodity market reports.

### Sample Data

First, let's create some sample documents to work with.

In [ ]:
# Sample commodity reports
sample_reports = [
    {
        'id': 'eia_wpsr_2024_03_15',
        'title': 'EIA Weekly Petroleum Status Report - March 15, 2024',
        'date': '2024-03-15',
        'source': 'EIA',
        'report_type': 'inventory',
        'content': '''U.S. commercial crude oil inventories (excluding those in the Strategic Petroleum Reserve) 
        decreased by 1.5 million barrels from the previous week. At 445.1 million barrels, U.S. crude oil 
        inventories are about 3% below the five-year average for this time of year. Total motor gasoline 
        inventories increased by 0.6 million barrels and are about 1% below the five-year average. 
        Distillate fuel inventories decreased by 0.9 million barrels and are about 6% below the five-year average. 
        Refinery inputs averaged 15.9 million barrels per day, representing a refinery utilization rate of 89.2%.'''
    },
    {
        'id': 'opec_omr_2024_03',
        'title': 'OPEC Monthly Oil Market Report - March 2024',
        'date': '2024-03-12',
        'source': 'OPEC',
        'report_type': 'market_outlook',
        'content': '''OPEC maintained its forecast for global oil demand growth in 2024 at 2.25 million barrels per day. 
        The forecast is supported by expectations of robust economic activity in China and India, as well as 
        continued recovery in air travel demand. Non-OPEC supply growth is projected at 1.5 million bpd, led by 
        increases in U.S. shale production and Canadian oil sands. OPEC+ production cuts of 2.2 million bpd remain 
        in effect through Q2 2024 to support market balance and price stability.'''
    },
    {
        'id': 'iea_omr_2024_03',
        'title': 'IEA Oil Market Report - March 2024',
        'date': '2024-03-14',
        'source': 'IEA',
        'report_type': 'market_outlook',
        'content': '''The IEA revised its 2024 global oil demand growth forecast down to 1.3 million barrels per day, 
        citing weaker-than-expected economic performance in developed economies. However, demand in emerging markets, 
        particularly in Asia, continues to show resilience. Global oil supply increased by 500,000 bpd month-on-month, 
        driven primarily by higher output from the United States, Brazil, and Guyana. The IEA projects the global oil 
        market will see a surplus of approximately 1 million bpd in the first half of 2024.'''
    },
    {
        'id': 'api_weekly_2024_03_13',
        'title': 'API Weekly Statistical Bulletin - March 13, 2024',
        'date': '2024-03-13',
        'source': 'API',
        'report_type': 'inventory',
        'content': '''The American Petroleum Institute reported a crude oil inventory draw of 3.2 million barrels for the 
        week ending March 8. This compared to analyst expectations of a 1.8 million barrel draw. Gasoline stocks 
        increased by 1.4 million barrels, while distillate inventories fell by 1.2 million barrels. Cushing, Oklahoma 
        inventories, the delivery point for WTI futures, decreased by 0.8 million barrels to 28.5 million barrels.'''
    },
    {
        'id': 'usda_wasde_2024_03',
        'title': 'USDA World Agricultural Supply and Demand Estimates - March 2024',
        'date': '2024-03-08',
        'source': 'USDA',
        'report_type': 'agricultural',
        'content': '''U.S. corn production for 2023/24 is forecast at 15.3 billion bushels, up from last month. 
        Soybean production is estimated at 4.16 billion bushels. Global wheat production is projected at 783 million 
        metric tons. Weather conditions in the U.S. Midwest have been favorable for planting. Corn ending stocks are 
        forecast at 2.2 billion bushels, above trade expectations. Export forecasts were revised upward based on 
        strong demand from China and Mexico.'''
    }
]

# Create a DataFrame
df_reports = pd.DataFrame(sample_reports)
print(f"Created {len(df_reports)} sample reports")
df_reports[['title', 'date', 'source']].head()

### Knowledge Bank Configuration

Key configuration options:

1. **Chunking Strategy**
   - `fixed`: Fixed size chunks (simple, consistent)
   - `recursive`: Respects paragraphs and sentences (better quality)
   - `semantic`: AI-powered boundaries (most expensive)

2. **Chunk Size**
   - Small (200-300 tokens): More precise, more chunks to search
   - Medium (500-600 tokens): Balanced (recommended)
   - Large (800-1000 tokens): More context, less precise

3. **Embedding Model**
   - `text-embedding-3-small`: Fast, cheap, 1536 dimensions
   - `text-embedding-3-large`: Higher quality, 3072 dimensions
   - `voyage-2`: Alternative provider

In [ ]:
# Create Knowledge Bank (simulated - actual API may differ)
# In real Dataiku, you'd use the UI or API

class SimpleKnowledgeBank:
    """
    Simplified Knowledge Bank for demonstration.
    In production, use dataiku.knowledge_bank.KnowledgeBank
    """
    
    def __init__(self, name: str, embedding_connection: str = 'openai-embeddings'):
        self.name = name
        self.embedding_connection = embedding_connection
        self.documents = []
        self.chunks = []
        print(f"Created Knowledge Bank: {name}")
    
    def add_documents(self, documents: List[Dict]):
        """Add documents with metadata."""
        for doc in documents:
            self.documents.append({
                'id': doc['id'],
                'content': doc['content'],
                'metadata': {
                    'title': doc.get('title'),
                    'date': doc.get('date'),
                    'source': doc.get('source'),
                    'report_type': doc.get('report_type')
                }
            })
        print(f"Added {len(documents)} documents")
    
    def chunk_documents(self, chunk_size: int = 500, overlap: int = 50):
        """Chunk documents into smaller pieces."""
        self.chunks = []
        
        for doc in self.documents:
            # Simple chunking by words (real implementation uses tokens)
            words = doc['content'].split()
            
            for i in range(0, len(words), chunk_size - overlap):
                chunk_words = words[i:i + chunk_size]
                if len(chunk_words) > 50:  # Skip very small chunks
                    self.chunks.append({
                        'text': ' '.join(chunk_words),
                        'metadata': doc['metadata'],
                        'document_id': doc['id'],
                        'chunk_index': len(self.chunks)
                    })
        
        print(f"Created {len(self.chunks)} chunks from {len(self.documents)} documents")
        return len(self.chunks)
    
    def build(self):
        """Build the vector index."""
        print(f"Building vector index for {len(self.chunks)} chunks...")
        print("✓ Index built successfully")
    
    def search(self, query: str, top_k: int = 5, filters: Dict = None) -> List[Dict]:
        """Search for relevant chunks (simplified - actual uses vector similarity)."""
        # Simplified search using keyword matching
        # Real implementation uses vector similarity
        query_lower = query.lower()
        results = []
        
        for chunk in self.chunks:
            # Apply filters if specified
            if filters:
                if not all(chunk['metadata'].get(k) == v for k, v in filters.items()):
                    continue
            
            # Simple relevance score (count matching words)
            chunk_lower = chunk['text'].lower()
            score = sum(word in chunk_lower for word in query_lower.split())
            
            if score > 0:
                results.append({
                    'text': chunk['text'],
                    'score': score / len(query_lower.split()),
                    'metadata': chunk['metadata']
                })
        
        # Sort by score and return top_k
        results.sort(key=lambda x: x['score'], reverse=True)
        return results[:top_k]

# Create and populate Knowledge Bank
kb = SimpleKnowledgeBank(name='commodity_reports_kb')
kb.add_documents(sample_reports)
num_chunks = kb.chunk_documents(chunk_size=500, overlap=50)
kb.build()

print(f"\nKnowledge Bank ready with {num_chunks} chunks")

### Exercise 1.1: Test Basic Retrieval

**Task**: Query the Knowledge Bank and examine the retrieved chunks.

In [ ]:
# Test queries
test_queries = [
    "What was the crude oil inventory change?",
    "What is OPEC's demand forecast?",
    "Tell me about corn production"
]

for query in test_queries:
    print(f"\n{'='*60}")
    print(f"Query: {query}")
    print('='*60)
    
    results = kb.search(query, top_k=3)
    
    for i, result in enumerate(results, 1):
        print(f"\nResult {i} (score: {result['score']:.2f}):")
        print(f"Source: {result['metadata']['source']} - {result['metadata']['date']}")
        print(f"Text: {result['text'][:200]}...")

# Auto-graded test
results = kb.search("crude oil inventory", top_k=3)
assert len(results) > 0, "Should return results"
assert 'text' in results[0], "Results should contain text"
assert 'metadata' in results[0], "Results should contain metadata"
print("\n✓ Exercise 1.1 passed!")

## Exercise 2: Chunking Strategies

Chunking strategy significantly impacts retrieval quality.

### Chunking Tradeoffs

| Strategy | Pros | Cons | Best For |
|----------|------|------|----------|
| Small chunks (200-300) | Precise matching | May lose context | Factual Q&A |
| Medium chunks (500-600) | Balanced | General purpose | Most use cases |
| Large chunks (800-1000) | More context | Less precise | Complex reasoning |

### Overlap Importance

Chunk overlap prevents information from being split across boundaries:
- No overlap: Risk splitting concepts
- 10-20% overlap: Recommended
- >30% overlap: Redundant, wastes storage

In [ ]:
# Compare chunking strategies

def compare_chunking(documents: List[Dict], strategies: List[Dict]) -> pd.DataFrame:
    """
    Compare different chunking configurations.
    """
    results = []
    
    for strategy in strategies:
        kb_temp = SimpleKnowledgeBank(f"kb_{strategy['name']}")
        kb_temp.add_documents(documents)
        num_chunks = kb_temp.chunk_documents(
            chunk_size=strategy['chunk_size'],
            overlap=strategy['overlap']
        )
        
        # Calculate average chunk length
        avg_length = sum(len(c['text'].split()) for c in kb_temp.chunks) / len(kb_temp.chunks)
        
        results.append({
            'strategy': strategy['name'],
            'chunk_size': strategy['chunk_size'],
            'overlap': strategy['overlap'],
            'num_chunks': num_chunks,
            'avg_chunk_words': int(avg_length)
        })
    
    return pd.DataFrame(results)

# Test different strategies
strategies = [
    {'name': 'small_no_overlap', 'chunk_size': 200, 'overlap': 0},
    {'name': 'small_overlap', 'chunk_size': 200, 'overlap': 40},
    {'name': 'medium_overlap', 'chunk_size': 500, 'overlap': 100},
    {'name': 'large_overlap', 'chunk_size': 800, 'overlap': 160},
]

comparison = compare_chunking(sample_reports, strategies)
print("\nChunking Strategy Comparison:")
print(comparison.to_string(index=False))

# Insight
print("\n📊 Insight:")
print(f"- Smaller chunks create {comparison.iloc[0]['num_chunks']} pieces")
print(f"- Larger chunks create {comparison.iloc[-1]['num_chunks']} pieces")
print(f"- More chunks = more precise matching but higher search cost")

### Exercise 2.1: Find Optimal Chunk Size

**Task**: Test different chunk sizes on a specific query and determine which works best.

In [ ]:
def evaluate_chunking(query: str, documents: List[Dict], chunk_sizes: List[int]) -> pd.DataFrame:
    """
    Evaluate which chunk size works best for a query.
    
    YOUR TASK:
    - Create KBs with different chunk sizes
    - Query each one
    - Compare: num results, avg score, top result quality
    """
    results = []
    
    # YOUR CODE HERE
    
    return pd.DataFrame(results)

# Test on a specific query
test_query = "What is the global oil demand growth forecast?"
chunk_sizes_to_test = [200, 400, 600, 800]

eval_results = evaluate_chunking(test_query, sample_reports, chunk_sizes_to_test)

# YOUR CODE HERE: Print and analyze results

# Auto-graded test
assert len(eval_results) == len(chunk_sizes_to_test), "Should test all chunk sizes"
print("\n✓ Exercise 2.1 passed!")

## Exercise 3: Metadata and Filtered Retrieval

Metadata enables powerful filtering:
- **Date filters**: "Only reports from last month"
- **Source filters**: "Only EIA reports"
- **Type filters**: "Only inventory reports"

This is crucial for production RAG systems where you need:
- Temporal relevance (recent data)
- Source credibility (trusted publishers)
- Content type (the right kind of report)

In [ ]:
# Demonstrate filtered retrieval

query = "What is the inventory level?"

print("=== UNFILTERED SEARCH ===")
unfiltered = kb.search(query, top_k=3)
for i, result in enumerate(unfiltered, 1):
    print(f"\n{i}. Source: {result['metadata']['source']} ({result['metadata']['report_type']})")
    print(f"   Date: {result['metadata']['date']}")
    print(f"   Score: {result['score']:.2f}")

print("\n" + "="*60)
print("=== FILTERED: Only EIA inventory reports ===")
filtered = kb.search(query, top_k=3, filters={'source': 'EIA', 'report_type': 'inventory'})
for i, result in enumerate(filtered, 1):
    print(f"\n{i}. Source: {result['metadata']['source']} ({result['metadata']['report_type']})")
    print(f"   Date: {result['metadata']['date']}")
    print(f"   Score: {result['score']:.2f}")

print("\n📊 Notice how filtering ensures relevance!")

### Exercise 3.1: Design Metadata Schema

**Task**: Design a metadata schema for a commodity trading firm's document library.

Consider:
- What documents do they have? (reports, news, analysis, etc.)
- What filters would be useful? (date, commodity, source, etc.)
- What enables good retrieval? (tags, categories, etc.)

In [ ]:
# YOUR CODE HERE
# Define a metadata schema as a dictionary

metadata_schema = {
    'required_fields': [
        # YOUR CODE HERE
    ],
    'optional_fields': [
        # YOUR CODE HERE
    ],
    'field_types': {
        # YOUR CODE HERE
        # e.g., 'date': 'datetime', 'source': 'string'
    },
    'filterable_fields': [
        # YOUR CODE HERE
        # Which fields should support filtering?
    ]
}

# Create sample documents with your schema
sample_docs_with_metadata = [
    # YOUR CODE HERE
    # At least 3 sample documents following your schema
]

# Validate schema
print("Metadata Schema:")
print(json.dumps(metadata_schema, indent=2))

print("\nSample Document Metadata:")
if sample_docs_with_metadata:
    print(json.dumps(sample_docs_with_metadata[0], indent=2))

# Auto-graded tests
assert 'required_fields' in metadata_schema, "Schema must define required fields"
assert len(metadata_schema['required_fields']) >= 3, "Should have at least 3 required fields"
assert 'filterable_fields' in metadata_schema, "Schema must specify filterable fields"
assert len(sample_docs_with_metadata) >= 3, "Should create at least 3 sample documents"

print("\n✓ Exercise 3.1 passed!")

## Exercise 4: Knowledge Bank Maintenance

Production Knowledge Banks need regular maintenance:
1. **Incremental updates**: Add new documents without rebuilding
2. **Document deletion**: Remove outdated content
3. **Reindexing**: Rebuild when embeddings change
4. **Monitoring**: Track KB size, query performance

In [ ]:
# Extend SimpleKnowledgeBank with maintenance operations

class MaintainableKB(SimpleKnowledgeBank):
    """Knowledge Bank with maintenance capabilities."""
    
    def __init__(self, name: str, embedding_connection: str = 'openai-embeddings'):
        super().__init__(name, embedding_connection)
        self.stats = {
            'total_adds': 0,
            'total_deletes': 0,
            'total_queries': 0,
            'last_updated': None
        }
    
    def add_documents_incremental(self, documents: List[Dict]):
        """Add documents without rebuilding entire index."""
        from datetime import datetime
        
        print(f"Adding {len(documents)} documents incrementally...")
        self.add_documents(documents)
        
        # Rechunk only new documents
        new_chunks = self.chunk_documents()
        
        self.stats['total_adds'] += len(documents)
        self.stats['last_updated'] = datetime.now().isoformat()
        
        print(f"✓ Added {new_chunks} new chunks")
    
    def delete_documents(self, document_ids: List[str]):
        """Remove documents by ID."""
        from datetime import datetime
        
        before = len(self.documents)
        self.documents = [d for d in self.documents if d['id'] not in document_ids]
        removed = before - len(self.documents)
        
        # Rebuild chunks
        self.chunk_documents()
        
        self.stats['total_deletes'] += removed
        self.stats['last_updated'] = datetime.now().isoformat()
        
        print(f"✓ Removed {removed} documents")
    
    def get_stats(self) -> Dict:
        """Return KB statistics."""
        return {
            **self.stats,
            'total_documents': len(self.documents),
            'total_chunks': len(self.chunks),
            'avg_chunks_per_doc': len(self.chunks) / len(self.documents) if self.documents else 0
        }

# Test maintenance operations
kb_main = MaintainableKB('production_kb')
kb_main.add_documents(sample_reports[:3])
kb_main.chunk_documents()
kb_main.build()

print("\nInitial stats:")
print(json.dumps(kb_main.get_stats(), indent=2))

# Add more documents
kb_main.add_documents_incremental(sample_reports[3:])

print("\nAfter adding documents:")
print(json.dumps(kb_main.get_stats(), indent=2))

# Delete old documents
kb_main.delete_documents(['eia_wpsr_2024_03_15'])

print("\nAfter deletion:")
print(json.dumps(kb_main.get_stats(), indent=2))

### Exercise 4.1: Implement Scheduled Refresh

**Task**: Design a refresh strategy for a Knowledge Bank that ingests daily reports.

Consider:
- How often to refresh?
- Incremental vs full rebuild?
- How to handle duplicates?
- What to monitor?

In [ ]:
def design_refresh_strategy() -> Dict:
    """
    Design a refresh strategy for daily commodity reports.
    
    Return a dictionary with:
    - refresh_frequency: how often to refresh
    - refresh_type: 'incremental' or 'full'
    - duplicate_handling: how to handle duplicates
    - retention_policy: how long to keep old documents
    - monitoring_metrics: what to track
    """
    # YOUR CODE HERE
    strategy = {
        'refresh_frequency': '',  # e.g., 'daily', 'hourly'
        'refresh_type': '',       # 'incremental' or 'full'
        'duplicate_handling': '', # how to detect and handle duplicates
        'retention_policy': '',   # e.g., 'keep 90 days'
        'monitoring_metrics': []  # what to monitor
    }
    
    return strategy

# Implement a simple refresh function
def refresh_knowledge_bank(kb: MaintainableKB, new_documents: List[Dict], strategy: Dict):
    """
    Refresh KB according to strategy.
    
    YOUR TASK:
    - Check for duplicates
    - Add new documents
    - Apply retention policy
    - Log results
    """
    from datetime import datetime, timedelta
    
    # YOUR CODE HERE
    pass

# Test your strategy
strategy = design_refresh_strategy()
print("Refresh Strategy:")
print(json.dumps(strategy, indent=2))

# Auto-graded tests
assert 'refresh_frequency' in strategy, "Must specify refresh frequency"
assert 'duplicate_handling' in strategy, "Must specify duplicate handling"
assert 'monitoring_metrics' in strategy, "Must specify monitoring metrics"
assert len(strategy['monitoring_metrics']) > 0, "Should monitor at least one metric"

print("\n✓ Exercise 4.1 passed!")

## Summary

Congratulations! You've mastered Knowledge Bank creation. Key takeaways:

1. **Knowledge Banks ground LLMs in facts** - Reduces hallucination, enables citations
2. **Chunking strategy matters** - Balance precision vs context
3. **Metadata enables filtering** - Critical for production relevance
4. **Maintenance is ongoing** - Plan for incremental updates and monitoring
5. **Test retrieval quality** - Different strategies for different use cases

## Best Practices

- Start with 500-token chunks, 10% overlap
- Always include metadata (date, source, type)
- Test retrieval on real queries before production
- Monitor KB size and query latency
- Refresh regularly (daily for time-sensitive content)

## Next Steps

- **Notebook 02**: Build complete RAG workflows with generation
- **Module 3**: Deploy RAG as production API
- **Module 4**: Monitor RAG performance in production

In [ ]:
key_takeaways([
    "Core concept from this notebook demonstrated with working code",
    "Key parameters and their effects on results",
    "When to apply this technique vs alternatives"
])